In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardizedScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.datasets import load_iris

# Load dataset
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
y_true = iris.target  # For validation only

# Data Exploration
plt.figure(figsize=(10, 6))
sns.scatterplot(x=df['petal length (cm)'], y=df['petal width (cm)'], hue=y_true, palette='deep')
plt.title('Petal Length vs. Petal Width (True Species)')
plt.xlabel('Petal Length (cm)')
plt.ylabel('Petal Width (cm)')
plt.savefig('iris_plot.png', bbox_inches='tight')
plt.close()

# Data Preparation
df['PetalArea'] = df['petal length (cm)'] * df['petal width (cm)']
scaler = StandardScaler()
X = scaler.fit_transform(df)

# Train and evaluate clustering models
models = {
    'K-Means': KMeans(n_clusters=3, random_state=42),
    'Hierarchical Clustering': AgglomerativeClustering(n_clusters=3, linkage='ward'),
    'DBSCAN': DBSCAN(eps=0.5, min_samples=5)
}
results = []
for name, model in models.items():
    labels = model.fit_predict(X)
    sil_score = silhouette_score(X, labels) if -1 not in labels else 0.0  # DBSCAN outlier handling
    ari_score = adjusted_rand_score(y_true, labels)
    results.append({
        'Model': name,
        'Silhouette Score': sil_score,
        'Adjusted Rand Index': ari_score
    })
    if name == 'K-Means':
        plt.figure(figsize=(10, 6))
        sns.scatterplot(x=df['petal length (cm)'], y=df['petal width (cm)'], hue=labels, palette='deep')
        plt.title('Petal Length vs. Petal Width by Cluster (K-Means)')
        plt.xlabel('Petal Length (cm)')
        plt.ylabel('Petal Width (cm)')
        plt.savefig('iris_plot.png', bbox_inches='tight')
        plt.close()

# Display results
results_df = pd.DataFrame(results)
print(results_df)

# Feature Importance (approximated via K-Means cluster centers)
kmeans = models['K-Means']
feature_importance = pd.Series(np.abs(kmeans.cluster_centers_).mean(axis=0), index=df.columns).sort_values(ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(x=feature_importance.values, y=feature_importance.index)
plt.title('Feature Contributions to Clustering (K-Means)')
plt.xlabel('Mean Absolute Cluster Center Value')
plt.ylabel('Feature')
plt.savefig('feature_importance.png', bbox_inches='tight')
plt.close()
